In [1]:
from pyomo.environ import *
import pandas as pd

In [2]:

Price = pd.read_csv('spot price.csv')['Spot Price']
readin_Hours = pd.read_csv('Hours.csv')['Hours']
Solar_Gen = pd.read_csv('Solar_gen.csv')['Solar Gen']

In [3]:
# Define the Solar PPA price in Pounds/MWh
SOLAR_PPA_PRICE = 30

# Define the maximum battery capacity in MWh
MAX_BATTERY_CAPACITY = 10

# Define the minimum battery capacity MWh
MIN_BATTERY_CAPACITY = 2

# Define the initial battery capacity in MWh
INITIAL_CAPACITY = MAX_BATTERY_CAPACITY

# Define the efficiency of the battery 
EFFICIENCY = 0.95

# Define the Maximum Power of the battery in MW
MAX_BATTERY_POWER = 5

# Define the Capital Cost of battery in Pounds/MWh
CAPITAL_COST = 200000
DISCOUNT_RATE = 0.03
YEARS = 15

ANNUALIZED_CAPITAL_COST = (CAPITAL_COST * (1+DISCOUNT_RATE)**YEARS * DISCOUNT_RATE)/((1+DISCOUNT_RATE)**YEARS-1)
ANNUALIZED_CAPITAL_COST

16753.316092457597

In [4]:
# Define the model
battery = AbstractModel()

instance = battery.create_instance()
# Set the time period
battery.Period = readin_Hours
battery.Spot_Price = Param(battery.Period, initialize=lambda m, i: Price[i-1])
battery.Solar_Gen = Param(battery.Period, initialize=lambda m, i: Solar_Gen[i-1])
battery.Charging_Price = Param(battery.Period, 
                                initialize=lambda m, i: Price[i-1] if Price[i-1] <= SOLAR_PPA_PRICE else SOLAR_PPA_PRICE,
                                mutable=True)



# Define the battery capacity
battery.Capacity = Var(battery.Period, within=NonNegativeReals)

# Define the battery charge power
battery.Charge_power = Var(battery.Period, within=NonNegativeReals)

# Define the battery discharge power
battery.Discharge_power = Var(battery.Period, within=NonNegativeReals)

battery.Solar_PPA_Gen = Var(battery.Period, initialize=lambda m, i: Solar_Gen[i-1])
battery.Charge_power_solar = Var(battery.Period, within=NonNegativeReals)


# Set constraints for the battery
# Defining capacity rule for the battery
def capacity_constraint(battery, i):
    # Assigning battery capacity at the beginning of optimization
    if i == min(battery.Period):
        return battery.Capacity[i] == INITIAL_CAPACITY
    return battery.Capacity[i] == battery.Capacity[i-1] + (battery.Charge_power[i-1] * EFFICIENCY) + (battery.Charge_power_solar[i-1] * EFFICIENCY) - battery.Discharge_power[i-1] / EFFICIENCY

#Make sure the battery does not charge above the limit
def over_charge(battery, i):
    if i > 1:
        return battery.Charge_power[i] + battery.Charge_power_solar[i] <= (MAX_BATTERY_CAPACITY - battery.Capacity[i-1])/ EFFICIENCY
    return Constraint.Skip
    
# Make sure the battery discharge the amount it actually has
def over_discharge(battery, i):
    if i > 1:
        return battery.Discharge_power[i] <= (battery.Capacity[i-1] - MIN_BATTERY_CAPACITY )*EFFICIENCY
    return Constraint.Skip

# Make sure the battery does not charge more than Battery power
def power_charge(battery, i):
    return battery.Charge_power[i] + battery.Charge_power_solar[i] <= MAX_BATTERY_POWER

# Make sure the battery does not discharge more than Battery power
def power_discharge(battery, i):
    return battery.Discharge_power[i] <= MAX_BATTERY_POWER

# Make sure the battery does not charge more than solar generation while charging from Solar PV
def Solar_charge_rule(battery, i):
    if i>1:
        return battery.Charge_power_solar[i] <= battery.Solar_Gen[i] 
    return Constraint.Skip


# To calculate the solar generation after deducting the energy to charge battery
def Solar_PPA_rule(battery, i):
        return battery.Solar_PPA_Gen[i] == battery.Solar_Gen[i] - battery.Charge_power_solar[i]

       
   


# Add the constraints to the model
battery.capacity_constraint = Constraint(battery.Period, rule=capacity_constraint)
battery.over_charge = Constraint(battery.Period, rule=over_charge)
battery.over_discharge = Constraint(battery.Period, rule=over_discharge)
battery.power_charge = Constraint(battery.Period, rule=power_charge)
battery.power_discharge = Constraint(battery.Period, rule=power_discharge)
battery.Solar_charge_rule = Constraint(battery.Period, rule=Solar_charge_rule)
battery.Solar_PPA_rule = Constraint(battery.Period, rule=Solar_PPA_rule)






CORRECTION_FACTOR = 1
COST_PER_CYCLE = (ANNUALIZED_CAPITAL_COST)/(5000*MAX_BATTERY_CAPACITY*(1-0.8)*EFFICIENCY)




# Define the objective function

#battery.Solar_PPA_Gen[i] * SOLAR_PPA_PRICE 
def battery_objective_rule(battery):
    return sum((battery.Solar_PPA_Gen[i] * SOLAR_PPA_PRICE) + (battery.Discharge_power[i] * battery.Spot_Price[i]) - (battery.Charge_power[i] * battery.Spot_Price[i]) - (battery.Charge_power_solar[i] * SOLAR_PPA_PRICE) - battery.COST_PER_HOUR[i] for i in battery.Period)


battery.objective = Objective(rule=battery_objective_rule, sense=maximize)

    # Create an instance of the model
instance = battery.create_instance()
opt = SolverFactory('glpk')
opt.solve(instance)





{'Problem': [{'Name': 'unknown', 'Lower bound': 755252.270281243, 'Upper bound': 755252.270281243, 'Number of objectives': 1, 'Number of constraints': 61318, 'Number of variables': 43801, 'Number of nonzeros': 140151, 'Sense': 'maximize'}], 'Solver': [{'Status': 'ok', 'Termination condition': 'optimal', 'Statistics': {'Branch and bound': {'Number of bounded subproblems': 0, 'Number of created subproblems': 0}}, 'Error rc': 0, 'Time': 16.58661651611328}], 'Solution': [OrderedDict([('number of solutions', 0), ('number of solutions displayed', 0)])]}

In [5]:
obj_val = value(instance.objective())
print("Total Profit: ", obj_val)


Total Profit:  755252.2702812648


In [6]:
total = 0
for i in range(2, 8760):
    total += value(instance.Discharge_power[i])
    
#print("BatteryOM COST:", BATTERY_OM_COST)
#print("Annual cost of discharging battery", COST_PER_CYCLE*total)
Error = ((BATTERY_OM_COST - COST_PER_CYCLE*total) / BATTERY_OM_COST)*100
Error
#print("correction factor:", CORRECTION_FACTOR)
print("discharging_cycles:", discharging_cycles)

NameError: name 'BATTERY_OM_COST' is not defined

In [ ]:
instance.display()

In [ ]:
solver_status = opt.solve(instance)
print(solver_status.solver.status)

In [ ]:
from pyomo.opt import SolverStatus, TerminationCondition

results = SolverFactory("glpk").solve(instance, tee=True)

if (results.solver.status == SolverStatus.ok) and (results.solver.termination_condition == TerminationCondition.optimal):
    instance.solutions.store_to(results)
    print("Optimal solution found!")

In [ ]:

print(battery.Solar_Gen[i])

In [ ]:
    for i in range(2, 8760):
        # Check if the battery is charging more than 1% of its total capacity
        if instance.Spot_Price[i].value > instance.SOLAR_PPA_PRICE.value:
            instance.Solar_Gen[i].value == instance.Solar_Gen[i].value - instance.Charge_power[i].value
            instance.Solar_Gen[i].value


In [7]:
# Store the optimization results in a Pandas DataFrame
results = pd.DataFrame({'Period': [i for i in instance.Period],
                        'Charge_power': [value(instance.Charge_power[i]) for i in battery.Period],
                        'Discharge_power': [value(instance.Discharge_power[i]) for i in battery.Period],
                       'Battery Capacity at each hour': [value(instance.Capacity[i]) for i in battery.Period],
                       'Solar PPA Gen': [value(instance.Solar_PPA_Gen[i]) for i in battery.Period],
                       'Solar_charge': [value(instance.Charge_power_solar[i]) for i in battery.Period]})

# Write the DataFrame to an excel file
results.to_excel('battery_optimization_results_adavnced30ppa.xlsx', index=False)

In [ ]:
for v in battery.component_objects(Var):
    for index in v:
        print(f"{v.name}[(battery.Charge_power[i])] = {value((battery.Charge_power[i]))}")

In [ ]:
    if abs((BATTERY_OM_COST - COST_PER_CYCLE*sum(value(instance.Discharge_power[i]) for i in battery.Period)) / BATTERY_OM_COST) <= 0.05:
            battery.del_component('instance')
            break

In [ ]:
 for iteration in range(1, 5):
    # create the solver object and specify the solver to be used
    opt = SolverFactory('glpk')
    opt.solve(instance) 
    # Create a variable to store the charging cycles
    charging_cycles = 0

    # Create a variable to store the discharging cycles
    discharging_cycles = 0
    # Loop through the hours in the optimization result
    for i in range(2, 8760):
        # Check if the battery is charging more than 1% of its total capacity
        if instance.Capacity[i].value==10 and instance.Capacity[i].value - instance.Capacity[i-1].value>0.2*MAX_BATTERY_CAPACITY:
            charging_cycles += 1
        # Check if the battery is discharging more than 1% of its total capacity
        elif instance.Capacity[i].value==2 and instance.Capacity[i-1].value - instance.Capacity[i].value >0.2*MAX_BATTERY_CAPACITY:
            discharging_cycles += 1

    BATTERY_OM_COST =(ANNUALIZED_CAPITAL_COST*MAX_BATTERY_CAPACITY*discharging_cycles)/5000   
    CORRECTION_FACTOR = BATTERY_OM_COST/(COST_PER_CYCLE*sum(value(instance.Discharge_power[i])*CORRECTION_FACTOR for i in battery.Period))
        
    if CORRECTION_FACTOR <= 1.05:
            battery.del_component('objective')
            break